In [1]:
import math
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.cm import ScalarMappable
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.ticker import FuncFormatter

try:
    from matplotlib_map_utils import north_arrow, scale_bar  # type: ignore[import-not-found]
    HAS_MMU = True
except Exception:
    HAS_MMU = False

plt.rcParams.update(
    {
        'figure.dpi': 120,
        'savefig.dpi': 300,
        'font.size': 13,
        'axes.titlesize': 19,
        'axes.labelsize': 13,
        'legend.fontsize': 13,
        'legend.title_fontsize': 14,
        'axes.edgecolor': '#333333',
        'axes.linewidth': 0.8,
    }
)

PATH_NEW_BUILDINGS = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\LoD2\LoD2_2025_new_buildings_thr_h11m.gpkg"
PATH_EXISTING_BUILDINGS = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Output\Analysis_w_LoD2\LoD2\LoD2_2025_existing_thr_h11m.gpkg"
PATH_LANDKREISE = r"C:\Users\agz90fk\Documents\Masterarbeit\03_Daten\Input\Verwaltungsgebiete\vg250-ew_12-31.utm32s.gpkg.ebenen\vg250-ew_ebenen_1231\DE_VG250.gpkg"
LAYER_LANDKREISE = 'v_vg250_krs'

PATH_OUTPUT_DIR = Path(r"C:\Users\agz90fk\Documents\Masterarbeit\06_Abbildungen")
PATH_OUTPUT_CSV = PATH_OUTPUT_DIR / 'new_vs_existing_buildings_per_landkreis.csv'
PATH_OUTPUT_RANKED_CSV = PATH_OUTPUT_DIR / 'new_vs_existing_buildings_per_landkreis_ranked.csv'
PATH_OUTPUT_MAP = PATH_OUTPUT_DIR / 'new_to_existing_percent_bayern_map.jpg'

COL_ARS = 'Regionalschlüssel_ARS'
COL_GEN = 'GeografischerName_GEN'
COL_LK_KEY = 'LK_KEY'
COL_LK_NAME = 'LK_NAME'
COL_GEMEINDE = 'Gemeindeschluessel'
BAYERN_ARS_PREFIX = '09'
TARGET_CRS = 'EPSG:25832'
CRS_NOTE = 'CRS: EPSG:25832 - ETRS89 / UTM zone 32N'
GRID_STEP_M = 50000
MAP_PADDING_M = 10000
COLOR_THEME = ['#fff0f3', '#ffccd5',  '#ffb3c1',  '#ff8fa3',  '#ff758f', '#ff758f', '#ff4d6d', "#c9184a", "#a4133c","#800f2f", "#590d22"]
CUSTOM_CMAP = LinearSegmentedColormap.from_list('custom_orange_brown', COLOR_THEME)


def load_landkreise_bayern() -> gpd.GeoDataFrame:
    gdf = gpd.read_file(PATH_LANDKREISE, layer=LAYER_LANDKREISE)
    if COL_ARS not in gdf.columns:
        ars_candidates = [c for c in gdf.columns if 'ARS' in c or 'ars' in c]
        if not ars_candidates:
            raise ValueError('No ARS column found in Landkreis layer.')
        ars_col = ars_candidates[0]
    else:
        ars_col = COL_ARS

    gdf = gdf[gdf[ars_col].astype(str).str.startswith(BAYERN_ARS_PREFIX)].copy()
    gdf[COL_LK_KEY] = gdf[ars_col].astype(str).str[:5]
    gdf[COL_LK_NAME] = gdf[COL_GEN].astype(str) if COL_GEN in gdf.columns else gdf[COL_LK_KEY]
    gdf = gdf[[COL_LK_KEY, COL_LK_NAME, 'geometry']].dissolve(by=COL_LK_KEY, as_index=False)
    if gdf.crs is None or str(gdf.crs) != TARGET_CRS:
        gdf = gdf.to_crs(TARGET_CRS)
    return gdf


def load_buildings(path_buildings: str, target_crs, bbox):
    gdf = gpd.read_file(path_buildings, bbox=bbox, columns=[COL_GEMEINDE])
    if gdf.crs != target_crs:
        gdf = gdf.to_crs(target_crs)
    return gdf


def count_buildings_per_landkreis(gdf_buildings: gpd.GeoDataFrame, gdf_lk: gpd.GeoDataFrame) -> pd.Series:
    valid_lk = set(gdf_lk[COL_LK_KEY].astype(str))
    if COL_GEMEINDE in gdf_buildings.columns:
        lk_from_gemeinde = gdf_buildings[COL_GEMEINDE].astype(str).str[:5]
        lk_from_gemeinde = lk_from_gemeinde[lk_from_gemeinde.isin(valid_lk)]
        if not lk_from_gemeinde.empty:
            return lk_from_gemeinde.value_counts()

    joined = gpd.sjoin(
        gdf_buildings[['geometry']],
        gdf_lk[[COL_LK_KEY, 'geometry']],
        how='inner',
        predicate='intersects',
    )
    joined = joined[~joined.index.duplicated(keep='first')]
    return joined.groupby(COL_LK_KEY).size()


def build_result_if_needed(gdf_lk: gpd.GeoDataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    if PATH_OUTPUT_CSV.exists():
        result = pd.read_csv(PATH_OUTPUT_CSV, dtype={'ARS': str})
        result['ARS'] = result['ARS'].astype(str).str.zfill(5)
        if 'new_to_existing_percent' in result.columns:
            ranked_path = PATH_OUTPUT_RANKED_CSV
            if ranked_path.exists():
                ranked = pd.read_csv(ranked_path, dtype={'ARS': str})
                ranked['ARS'] = ranked['ARS'].astype(str).str.zfill(5)
            else:
                ranked = result.sort_values('new_to_existing_percent', ascending=False).reset_index(drop=True)
                ranked.insert(0, 'rank_new_vs_existing', ranked.index + 1)
                ranked.to_csv(ranked_path, index=False, encoding='utf-8-sig')
            return result, ranked

    bbox = tuple(gdf_lk.total_bounds)
    print('Loading new buildings...')
    gdf_new = load_buildings(PATH_NEW_BUILDINGS, gdf_lk.crs, bbox)
    print(f'  Loaded new buildings: {len(gdf_new)}')
    print('Loading existing buildings...')
    gdf_existing = load_buildings(PATH_EXISTING_BUILDINGS, gdf_lk.crs, bbox)
    print(f'  Loaded existing buildings: {len(gdf_existing)}')
    print('Counting buildings per district...')
    counts_new = count_buildings_per_landkreis(gdf_new, gdf_lk)
    counts_existing = count_buildings_per_landkreis(gdf_existing, gdf_lk)

    result = gdf_lk[[COL_LK_KEY, COL_LK_NAME, 'geometry']].drop_duplicates(subset=[COL_LK_KEY]).copy()
    result['area_km2'] = (result.geometry.area / 1_000_000).round(2)
    result['new_buildings_count'] = result[COL_LK_KEY].map(counts_new).fillna(0).astype(int)
    result['existing_buildings_count'] = result[COL_LK_KEY].map(counts_existing).fillna(0).astype(int)
    result['new_to_existing_ratio'] = result['new_buildings_count'] / result['existing_buildings_count'].replace(0, pd.NA)
    result['new_to_existing_percent'] = (result['new_to_existing_ratio'] * 100).round(2)
    result['new_share_of_total_percent'] = (
        result['new_buildings_count'] / (result['new_buildings_count'] + result['existing_buildings_count']).replace(0, pd.NA) * 100
    ).round(2)
    result['new_buildings_per_km2'] = (result['new_buildings_count'] / result['area_km2'].replace(0, pd.NA)).round(2)
    result['existing_buildings_per_km2'] = (result['existing_buildings_count'] / result['area_km2'].replace(0, pd.NA)).round(2)
    result['ARS'] = result[COL_LK_KEY].astype(str).str.zfill(5)
    result = result[
        [
            'ARS', COL_LK_NAME, 'area_km2', 'new_buildings_count', 'existing_buildings_count',
            'new_to_existing_ratio', 'new_to_existing_percent', 'new_share_of_total_percent',
            'new_buildings_per_km2', 'existing_buildings_per_km2',
        ]
    ].sort_values('ARS')

    ranked = result.sort_values('new_to_existing_percent', ascending=False).reset_index(drop=True)
    ranked.insert(0, 'rank_new_vs_existing', ranked.index + 1)

    PATH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    result.to_csv(PATH_OUTPUT_CSV, index=False, encoding='utf-8-sig')
    ranked.to_csv(PATH_OUTPUT_RANKED_CSV, index=False, encoding='utf-8-sig')
    return result, ranked


In [2]:
def add_matplotlib_grid(ax, bounds, step=GRID_STEP_M):
    minx, miny, maxx, maxy = bounds
    x_start = math.ceil(minx / step) * step
    x_end = math.floor(maxx / step) * step
    y_start = math.ceil(miny / step) * step
    y_end = math.floor(maxy / step) * step

    x_major = [x_start + i * step for i in range(int((x_end - x_start) / step) + 1)] if x_start <= x_end else []
    y_major = [y_start + i * step for i in range(int((y_end - y_start) / step) + 1)] if y_start <= y_end else []

    ax.set_xticks(x_major)
    ax.set_yticks(y_major)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{int(round(x / 1000))}'))
    ax.yaxis.set_major_formatter(FuncFormatter(lambda y, pos: f'{int(round(y / 1000))}'))

    if x_major and y_major:
        x_pts, y_pts = [], []
        for xv in x_major:
            for yv in y_major:
                x_pts.append(xv)
                y_pts.append(yv)
        ax.scatter(x_pts, y_pts, marker='+', s=30, linewidths=1.0, color='#8a8a8a', alpha=0.95, zorder=4, clip_on=True)

    ax.tick_params(axis='both', which='major', labelsize=11, length=0, colors='#9B9999')


def add_north_arrow(ax):
    if HAS_MMU:
        north_arrow(
            ax=ax,
            location='upper right',
            size='md',
            rotation={'degrees': 0},
            aob={
                'bbox_to_anchor': (0.985, 0.985),
                'bbox_transform': ax.transAxes,
                'pad': 0.06,
                'borderpad': 0.06,
                'facecolor': 'none',
                'edgecolor': 'none',
                'alpha': 1.0,
                'frameon': False,
            },
        )
        return

    ax.annotate(
        'N',
        xy=(0.965, 0.97),
        xytext=(0.965, 0.86),
        xycoords='axes fraction',
        textcoords='axes fraction',
        ha='center',
        va='center',
        fontsize=16,
        fontweight='bold',
        arrowprops=dict(arrowstyle='-|>', color='#222222', linewidth=1.4, shrinkA=0, shrinkB=0),
    )


def add_scale_bar(ax):
    if HAS_MMU:
        scale_bar(
            ax=ax,
            location='lower right',
            size='xs',
            style='ticks',
            bar={
                'projection': TARGET_CRS,
                'unit': 'km',
                'max': 50,
                'major_div': 2,
                'minor_div': 1,
                'minor_type': 'none',
                'reverse': False,
            },
            labels={
                'labels': ['0', '25', '50'],
                'style': 'major',
                'loc': 'below',
                'fontsize': 9,
            },
            units={'loc': 'text', 'label': 'km'},
            text={'fontfamily': 'sans-serif', 'fontsize': 9, 'textcolor': '#222222'},
            aob={
                'pad': 0.0,
                'borderpad': 1.0,
                'facecolor': 'none',
                'edgecolor': 'none',
                'alpha': 1.0,
                'frameon': False,
            },
        )
        return

    x0, x1 = ax.get_xlim()
    y0, y1 = ax.get_ylim()
    span_x = x1 - x0
    span_y = y1 - y0

    total_km = 50
    segment_km = 25
    bar_len = total_km * 1000
    segment_len = segment_km * 1000

    x_start = x1 - span_x * 0.25
    y_start = y0 + span_y * 0.032

    ax.plot([x_start, x_start + bar_len], [y_start, y_start], color='#222222', linewidth=1.3, zorder=5)
    tick_h = span_y * 0.006
    for x in [x_start, x_start + segment_len, x_start + bar_len]:
        ax.plot([x, x], [y_start - tick_h, y_start + tick_h], color='#222222', linewidth=1.0, zorder=5)

    txt_y = y_start + span_y * 0.011
    ax.text(x_start, txt_y, '0', ha='center', va='bottom', fontsize=9, color='#222222')
    ax.text(x_start + segment_len, txt_y, '25', ha='center', va='bottom', fontsize=9, color='#222222')
    ax.text(x_start + bar_len, txt_y, '50 km', ha='center', va='bottom', fontsize=9, color='#222222')


def add_scientific_frame(ax, gdf_base):
    ax.set_facecolor('#f1f1f1')
    minx, miny, maxx, maxy = gdf_base.total_bounds
    bounds = (minx - MAP_PADDING_M, miny - MAP_PADDING_M, maxx + MAP_PADDING_M, maxy + MAP_PADDING_M)
    ax.set_xlim(bounds[0], bounds[2])
    ax.set_ylim(bounds[1], bounds[3])
    ax.margins(0)
    add_matplotlib_grid(ax, bounds)
    ax.set_xlim(bounds[0], bounds[2])
    ax.set_ylim(bounds[1], bounds[3])
    ax.set_autoscale_on(False)
    ax.set_xlabel('Easting (km) - UTM 32N', fontsize=12, color='#555555')
    ax.set_ylabel('Northing (km) - UTM 32N', fontsize=12, color='#555555')
    ax.text(0.01, 0.01, CRS_NOTE, transform=ax.transAxes, ha='left', va='bottom', fontsize=11, color='#555555')
    for side in ['top', 'right']:
        ax.spines[side].set_visible(False)
    for side in ['left', 'bottom']:
        ax.spines[side].set_visible(True)
        ax.spines[side].set_color('#636262')
        ax.spines[side].set_linewidth(0.8)


def add_numeric_colorbar_right_margin(fig, sm, label):
    cax = fig.add_axes([0.84, 0.20, 0.025, 0.60])
    cbar = fig.colorbar(sm, cax=cax, orientation='vertical')
    cbar.outline.set_visible(False)
    cbar.ax.tick_params(labelsize=10, length=0, colors='#555555', pad=2)
    cbar.set_label(label, fontsize=12, color='#555555', labelpad=8)


def plot_numeric_choropleth_template(gdf_map: gpd.GeoDataFrame, value_col: str, output_path: Path, legend_title: str):
    fig, ax = plt.subplots(figsize=(8.6, 9.4))
    fig.subplots_adjust(right=0.80, left=0.07, top=0.92, bottom=0.08)

    base = gdf_map.copy()
    base.plot(ax=ax, column=value_col, cmap=CUSTOM_CMAP, edgecolor='white', linewidth=0.45, legend=False, zorder=2)

    sm = ScalarMappable(norm=Normalize(vmin=base[value_col].min(), vmax=base[value_col].max()), cmap=CUSTOM_CMAP)
    sm._A = []
    add_numeric_colorbar_right_margin(fig, sm, legend_title)

    add_north_arrow(ax)
    add_scale_bar(ax)
    # ax.set_title('New construction relative to existing stock at district level in Bavaria', pad=14, fontweight='bold')
    ax.set_aspect('equal')
    add_scientific_frame(ax, base)

    plt.savefig(output_path, bbox_inches='tight', facecolor='white', format='jpg')
    print(f'Done. Map written to: {output_path}')
    plt.close(fig)


def main():
    print('Loading Bavarian district boundaries...')
    gdf_lk = load_landkreise_bayern()
    result, ranked = build_result_if_needed(gdf_lk)

    map_gdf = gdf_lk[[COL_LK_KEY, COL_LK_NAME, 'geometry']].copy()
    map_gdf['ARS'] = map_gdf[COL_LK_KEY].astype(str).str.zfill(5)
    map_gdf = map_gdf.merge(result[['ARS', 'new_to_existing_percent']], on='ARS', how='left')
    plot_numeric_choropleth_template(
        gdf_map=map_gdf,
        value_col='new_to_existing_percent',
        output_path=PATH_OUTPUT_MAP,
        legend_title='New construction\nrelative to existing stock (%)',
    )

    print(f'Done. CSV written to: {PATH_OUTPUT_CSV}')
    print(f'Done. Ranked CSV written to: {PATH_OUTPUT_RANKED_CSV}')

    print('\nTop 10 districts by new_to_existing_percent:')
    print(
        ranked[
            ['rank_new_vs_existing', 'ARS', COL_LK_NAME, 'new_to_existing_percent', 'new_buildings_count', 'existing_buildings_count']
        ]
        .head(10)
        .to_string(index=False)
    )

    print('\nBottom 10 districts by new_to_existing_percent:')
    print(
        ranked[
            ['rank_new_vs_existing', 'ARS', COL_LK_NAME, 'new_to_existing_percent', 'new_buildings_count', 'existing_buildings_count']
        ]
        .tail(10)
        .to_string(index=False)
    )


if __name__ == '__main__':
    main()

Loading Bavarian district boundaries...
Loading new buildings...
  Loaded new buildings: 1811517
Loading existing buildings...
  Loaded existing buildings: 8271554
Counting buildings per district...
Done. Map written to: C:\Users\agz90fk\Documents\Masterarbeit\06_Abbildungen\new_to_existing_percent_bayern_map.jpg
Done. CSV written to: C:\Users\agz90fk\Documents\Masterarbeit\06_Abbildungen\new_vs_existing_buildings_per_landkreis.csv
Done. Ranked CSV written to: C:\Users\agz90fk\Documents\Masterarbeit\06_Abbildungen\new_vs_existing_buildings_per_landkreis_ranked.csv

Top 10 districts by new_to_existing_percent:
 rank_new_vs_existing   ARS                LK_NAME  new_to_existing_percent  new_buildings_count  existing_buildings_count
                    1 09776      Lindau (Bodensee)                    33.90                14806                     43678
                    2 09172   Berchtesgadener Land                    33.78                19665                     58220
              